In [1]:
print("ok")

ok


In [2]:
%pwd

'/home/hasan/dev/projects/Vantara-Customer-Intelligence-Platform/notebooks/data'

In [3]:
%cd ..

/home/hasan/dev/projects/Vantara-Customer-Intelligence-Platform/notebooks


In [4]:
%cd ..

/home/hasan/dev/projects/Vantara-Customer-Intelligence-Platform


In [5]:
import os 

In [6]:
config_path = os.getenv("CONFIG_PATH", "./config/config.yaml")
config_path 

'./config/config.yaml'

# Config.yaml

In [ ]:
source_uri="https://github.com/hasan-raza-01/Data/raw/main/online+retail+ii.zip"
raw_data_path="data/raw/data.csv"
schema_path="data/raw/schema.json"

In [8]:
print("source_uri:", source_uri)
print("raw_data_path:", raw_data_path)
print("schema_path:", schema_path)

source_uri: https://github.com/hasan-raza-01/Data/raw/main/online+retail+ii.zip
raw_data_path: data/raw/data.csv
schema_path: data/raw/schema.yaml


# src/utils/data.py

In [ ]:
from box import ConfigBox
import pandas as pd 
import sys, yaml, json


def generate_schema(
    df:pd.DataFrame
) -> dict[ str, tuple | dict[ str, str | dict[ str, str | int | float ] ] ]:
    """
    generates json schema of dataframe

    PARAMS: 
        - df (pd.DataFrame) : pandas data frame to generate json schema
    
    RETURNS: 
        dict[ str, tuple | dict[ str, str | dict[ str, str | int | float ] ] ]: json schema 
    """
    try:
        schema={}

        # add dataset shape
        schema["shape"]=df.shape 

        # add feature's stats
        for feature in df.columns: 
            name = feature.strip().lower()
            schema[name] = {
                "type": df[feature].dtype.name, 
                "stats": df[feature].describe().to_dict()
            }

        return schema
    except Exception as e: 
        raise CustomException(e, sys)

# src/utils/__init__.py

In [ ]:
def load_json(s:str | None = None, path:str | None = None) -> dict:
    """reads the data present inside the file provided in \'path\' variable

    Args:
        s (str | None): json string to reformat to dictionary, defaults to None
        path (str | None): path of the json file, defaults to None

    Returns:
        json: json of data inside file
    """
    try:
        if path and s: 
            raise ValueError("param \'s\' & \'path\' both are provided at same time, only 1 arg is supported at a time")
        elif path:
            with open(Path(path), 'r') as f:
                return json.load(f)
        else: 
            return json.loads(s)
    except Exception as e:
        raise CustomException(e, sys)

def dump_json(data:dict, path:str | None = None) -> None:
    """saves the dictoanary into json file

    Args:
        data (dict): dictionary data to save in form of json
        path (str | None): path to save the file, defaults to None
    """
    try:
        if path: 
            with open(Path(path), "w") as f:
                json.dump(data, fp=f, default=str, indent=4)
        else:
            return json.dumps(data, default=str, indent=4)
    except Exception as e:
        raise CustomException(e, sys)
def load_yaml(path:str) -> ConfigBox:
    """reads the yaml file available in path

    Args:
        path (str): path of the yaml file

    Returns:
        ConfigBox: dict["key"] = value --------->  dict.key = value
    """
    try:
        with open(Path(path), "r") as yaml_file_obj:
            return ConfigBox(yaml.safe_load(yaml_file_obj))
    except Exception as e:
        raise CustomException(e, sys)
    
def dump_yaml(content:dict, file_path:str) -> None:
    """saves the yaml file with provided content

    Args:
        content (any): content for the yaml file
        path (str): path to save the file
    """
    try:
        with open(Path(file_path), "w") as file:
            yaml.safe_dump(content, file)
    except Exception as e:
        raise CustomException(e, sys)

In [10]:
os.environ["CLEAN"]="false"
os.getenv("CLEAN")

'false'

# data/collection.py

In [11]:
from urllib.request import urlretrieve
from zipfile import ZipFile
from pathlib import Path
import pandas as pd
import sys, time

from src import CustomException, logger

In [ ]:
class DataCollector: 
    "DESCRIPTION: Collects data from internet using urllib.request.urlretrieve and persists in a local dir"
    def __init__(
            self, 
            uri: str, 
            raw_data_path: str | Path, 
            schema_path: str
    ):
        """
        PARAMS: 
        - uri: source uri of the file 
        - raw_data_path: path to save final dataframe
        - schema_path: path to save json schema 
        """
        self.uri = str(uri)
        self.raw_data_path = Path(raw_data_path)
        self.schema_path = Path(schema_path)
        self.path = self.raw_data_path.parent
        self.cleaning_paths = []

    def print_progress(self, count, block_size, total_size):
        "internal method used by 'download' method for printing download progress"
        downloaded = count * block_size
        percent = downloaded / total_size * 100
        if percent>100: 
            percent=100
        print(f"\rDownloaded: {percent:.2f}%", end='')

    def download(self): 
        "internal method used to download file from internet"
        try:  
            self.zip_path = self.path.joinpath(Path(self.uri).name)

            logger.info("downloading data...")
            start_time = time.time()

            # downlaods data from source uri and saved to zip file to provided path
            urlretrieve(self.uri, self.zip_path.absolute(), reporthook=self.print_progress)

            download_time = time.time()-start_time
            logger.info(f"download time: {download_time:.2f} sec")

        except Exception as e: 
            logger.error(e)
            if isinstance(e, CustomException):
                raise e
            else: 
                raise CustomException(e, sys)

    def extract(self): 
        "internal method used to extract downloaded .zip file"
        try: 
            logger.info("extracting file...")

            # extract downloaded zip file
            with ZipFile(self.zip_path.absolute(), "r") as zip_ref:
                zip_ref.extractall(self.path.absolute())
            self.cleaning_paths.append(self.zip_path.absolute())

            logger.info("extraction completed...")
            self.extracted_file_path = [self.path.joinpath(p).absolute() for p in os.listdir(self.path) if p.endswith(".xlsx")][0]
        except Exception as e: 
            logger.error(e)
            if isinstance(e, CustomException):
                raise e
            else: 
                raise CustomException(e, sys)

    def concat(self):
        "internal method used to concatenate all sheets to create a single dataframe"
        try: 
            logger.info("loading data...")
            data = pd.read_excel(self.extracted_file_path.absolute(), sheet_name=None)

            logger.info("concatenating sheets...")
            self.df = pd.concat(data.values(), ignore_index=True)

            try:
                logger.info("persisting concatenated data...")
                final_data_path=self.raw_data_path.absolute()
                self.df.to_csv(final_data_path, index=False)
                logger.info("data persisted successfully")
            except Exception as e: 
                logger.warning(f"failed to persist concatenated data, reason: {e}")

            # add extracted files to cleaning paths
            self.cleaning_paths.append(self.extracted_file_path)
        except Exception as e: 
            logger.error(e)
            if isinstance(e, CustomException):
                raise e
            else: 
                raise CustomException(e, sys)

    def schema(self): 
        "generates schema for final dataframe"
        try:
            logger.info("generating schema...") 
            schema = generate_schema(self.df)

            logger.info("saving schema...")
            dump_json(schema, self.schema_path)
            logger.info("schema saved successfully")
        except Exception as e: 
            logger.error(e)
            if isinstance(e, CustomException):
                raise e
            else: 
                raise CustomException(e, sys)

    def clean(self): 
        "deletes all files created throughout module process except final data file"
        try: 
            logger.info("cleaning all unwanted files...")

            # clean all unwanted files created through out process 
            for path in set(self.cleaning_paths):
                path.unlink()

            logger.info("cleaning completed")
        except Exception as e: 
            logger.error(e)
            if isinstance(e, CustomException):
                raise e
            else: 
                raise CustomException(e, sys)

    def collect(self): 
        "method that runs full data collection process"
        self.path.mkdir(parents=True, exist_ok=True)
        self.download()
        self.extract()
        self.concat()
        self.schema()
        if not os.getenv("CLEAN") or os.getenv("CLEAN").lower()!="false":
            self.clean()
        else: 
            logger.info(f"ENV var \'CLEAN\' is set to \'{os.getenv("CLEAN")}\'")

In [13]:
collector=DataCollector(
    source_uri, 
    raw_data_path, 
    schema_path
)

In [14]:
collector.collect()

Downloaded: 100.00%

In [15]:
import pandas as pd 

data = pd.read_excel("data/raw/online_retail_II.xlsx", sheet_name=None)
data.keys()

dict_keys(['Year 2009-2010', 'Year 2010-2011'])

In [16]:
df_1 = data[list(data.keys())[0]]
df_2 = data[list(data.keys())[1]]
df_1.shape, df_2.shape

((525461, 8), (541910, 8))

In [17]:
df_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 32.1+ MB


In [18]:
df_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[us]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [19]:
df_1.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [20]:
df_2.isnull().sum()

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

In [21]:
df = pd.read_csv("data/raw/data.csv")
df.shape

(1067371, 8)

In [22]:
print("{")
for feature in df.columns: 
    print(f"\t{feature}: {{\n\t\ttype: {df[feature].dtype.name}, {type(df[feature].dtype.name)}, \n\t\tstats: {df[feature].describe().to_dict()}\n\t}}")
print("}")

{
	Invoice: {
		type: str, <class 'str'>, 
		stats: {'count': 1067371, 'unique': 53628, 'top': '537434', 'freq': 1350}
	}
	StockCode: {
		type: str, <class 'str'>, 
		stats: {'count': 1067371, 'unique': 5305, 'top': '85123A', 'freq': 5829}
	}
	Description: {
		type: str, <class 'str'>, 
		stats: {'count': 1062989, 'unique': 5698, 'top': 'WHITE HANGING HEART T-LIGHT HOLDER', 'freq': 5918}
	}
	Quantity: {
		type: int64, <class 'str'>, 
		stats: {'count': 1067371.0, 'mean': 9.9388984711033, 'std': 172.7057940767534, 'min': -80995.0, '25%': 1.0, '50%': 3.0, '75%': 10.0, 'max': 80995.0}
	}
	InvoiceDate: {
		type: str, <class 'str'>, 
		stats: {'count': 1067371, 'unique': 47635, 'top': '2010-12-06 16:57:00', 'freq': 1350}
	}
	Price: {
		type: float64, <class 'str'>, 
		stats: {'count': 1067371.0, 'mean': 4.64938772741624, 'std': 123.5530587214632, 'min': -53594.36, '25%': 1.25, '50%': 2.1, '75%': 4.15, 'max': 38970.0}
	}
	Customer ID: {
		type: float64, <class 'str'>, 
		stats: {'count': 824

In [23]:
schema = generate_schema(df)
schema

{'shape': (1067371, 8),
 'invoice': {'type': 'str',
  'stats': {'count': 1067371, 'unique': 53628, 'top': '537434', 'freq': 1350}},
 'stockcode': {'type': 'str',
  'stats': {'count': 1067371, 'unique': 5305, 'top': '85123A', 'freq': 5829}},
 'description': {'type': 'str',
  'stats': {'count': 1062989,
   'unique': 5698,
   'top': 'WHITE HANGING HEART T-LIGHT HOLDER',
   'freq': 5918}},
 'quantity': {'type': 'int64',
  'stats': {'count': 1067371.0,
   'mean': 9.9388984711033,
   'std': 172.7057940767534,
   'min': -80995.0,
   '25%': 1.0,
   '50%': 3.0,
   '75%': 10.0,
   'max': 80995.0}},
 'invoicedate': {'type': 'str',
  'stats': {'count': 1067371,
   'unique': 47635,
   'top': '2010-12-06 16:57:00',
   'freq': 1350}},
 'price': {'type': 'float64',
  'stats': {'count': 1067371.0,
   'mean': 4.64938772741624,
   'std': 123.5530587214632,
   'min': -53594.36,
   '25%': 1.25,
   '50%': 2.1,
   '75%': 4.15,
   'max': 38970.0}},
 'customer id': {'type': 'float64',
  'stats': {'count': 8243

In [27]:
string = dump_json(schema)
print(string)

{
    "shape": [
        1067371,
        8
    ],
    "invoice": {
        "type": "str",
        "stats": {
            "count": 1067371,
            "unique": 53628,
            "top": "537434",
            "freq": 1350
        }
    },
    "stockcode": {
        "type": "str",
        "stats": {
            "count": 1067371,
            "unique": 5305,
            "top": "85123A",
            "freq": 5829
        }
    },
    "description": {
        "type": "str",
        "stats": {
            "count": 1062989,
            "unique": 5698,
            "top": "WHITE HANGING HEART T-LIGHT HOLDER",
            "freq": 5918
        }
    },
    "quantity": {
        "type": "int64",
        "stats": {
            "count": 1067371.0,
            "mean": 9.9388984711033,
            "std": 172.7057940767534,
            "min": -80995.0,
            "25%": 1.0,
            "50%": 3.0,
            "75%": 10.0,
            "max": 80995.0
        }
    },
    "invoicedate": {
        "type"

In [30]:
load_json(string)

{'shape': [1067371, 8],
 'invoice': {'type': 'str',
  'stats': {'count': 1067371, 'unique': 53628, 'top': '537434', 'freq': 1350}},
 'stockcode': {'type': 'str',
  'stats': {'count': 1067371, 'unique': 5305, 'top': '85123A', 'freq': 5829}},
 'description': {'type': 'str',
  'stats': {'count': 1062989,
   'unique': 5698,
   'top': 'WHITE HANGING HEART T-LIGHT HOLDER',
   'freq': 5918}},
 'quantity': {'type': 'int64',
  'stats': {'count': 1067371.0,
   'mean': 9.9388984711033,
   'std': 172.7057940767534,
   'min': -80995.0,
   '25%': 1.0,
   '50%': 3.0,
   '75%': 10.0,
   'max': 80995.0}},
 'invoicedate': {'type': 'str',
  'stats': {'count': 1067371,
   'unique': 47635,
   'top': '2010-12-06 16:57:00',
   'freq': 1350}},
 'price': {'type': 'float64',
  'stats': {'count': 1067371.0,
   'mean': 4.64938772741624,
   'std': 123.5530587214632,
   'min': -53594.36,
   '25%': 1.25,
   '50%': 2.1,
   '75%': 4.15,
   'max': 38970.0}},
 'customer id': {'type': 'float64',
  'stats': {'count': 8243